# 01-03 Tensor 数学运算：dim、keepdim、max 与点积

这一份只讲 Tensor 的常用数学运算，重点把 dim、keepdim、max 返回值、向量点积讲清楚。


In [ ]:
import torch
import numpy as np

torch.manual_seed(42)
print("torch version:", torch.__version__)


## 12. 常用数学运算

这一节先把常用运算讲清楚。你现在不要急着背所有函数，先记住两个问题：

1. 这个运算是在“每个元素上算”，还是在“某个维度上汇总”？
2. 如果函数里有 `dim`，它到底沿着哪一维算？

| 方法/符号 | 作用 | 常用参数 | 人话解释 |
|---|---|---|---|
| `x + y`、`x - y`、`x * y`、`x / y` | 逐元素运算 | 无，主要看 shape 是否相同或可广播 | 对应位置一个一个算 |
| `x @ y` | 矩阵乘法/向量点积 | 无，主要看矩阵形状是否能相乘 | 线性代数里的矩阵乘法 |
| `torch.matmul(x, y)` | 矩阵乘法/批量矩阵乘法 | `x`, `y` | 和 `@` 类似，函数写法更明确 |
| `x.sum(dim=None, keepdim=False)` | 求和 | `dim`、`keepdim` | 把所有数或某个维度上的数加起来 |
| `x.mean(dim=None, keepdim=False)` | 求平均值 | `dim`、`keepdim` | 把所有数或某个维度上的数求平均 |
| `x.max(dim=None, keepdim=False)` | 求最大值 | `dim`、`keepdim` | 找最大值；指定 `dim` 时还会返回最大值的位置索引 |

注意：`*` 是逐元素乘法，不是矩阵乘法。矩阵乘法用 `@` 或 `torch.matmul`。

In [ ]:
import torch

x = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

y = torch.tensor([
    [10.0, 20.0],
    [30.0, 40.0]
])

print("x =\n", x)
print("y =\n", y)

print("逐元素乘法 x * y：\n", x * y)
print("矩阵乘法 x @ y：\n", x @ y)
print("函数写法 torch.matmul(x, y)：\n", torch.matmul(x, y))

### 12.1 `dim` 到底是什么意思

`dim` 是 dimension 的缩写，意思是“维度”。

对二维 Tensor 来说：

```python
x.shape = [行数, 列数]
```

- `dim=0`：沿着第 0 维，也就是沿着“行的方向”往下算。结果会把多行压成一行，所以输出长度等于列数。
- `dim=1`：沿着第 1 维，也就是沿着“列的方向”横着算。结果会把多列压成一列，所以输出长度等于行数。

更直白一点：

| 写法 | 在二维表里怎么理解 | 输出形状 |
|---|---|---|
| `x.sum()` | 所有元素全加起来 | 标量 `[]` |
| `x.sum(dim=0)` | 每一列分别求和 | `[列数]` |
| `x.sum(dim=1)` | 每一行分别求和 | `[行数]` |

记忆方法：`dim` 指的是“被压缩掉的那一维”。

In [ ]:
import torch

x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

print("x =\n", x)
print("x.shape =", x.shape)

print("所有元素求和 x.sum() =", x.sum())
print("每一列求和 x.sum(dim=0) =", x.sum(dim=0))
print("每一行求和 x.sum(dim=1) =", x.sum(dim=1))

print("每一列平均 x.mean(dim=0) =", x.mean(dim=0))
print("每一行平均 x.mean(dim=1) =", x.mean(dim=1))

### 12.2 `keepdim` 是干什么的

`keepdim` 的意思是：求和、求平均、求最大值之后，要不要保留被压缩掉的维度。

默认是：

```python
keepdim=False
```

也就是压缩完之后，这个维度会消失。

如果设置：

```python
keepdim=True
```

这个维度不会消失，只是长度变成 1。

为什么要保留？主要是为了后面继续做广播运算时形状更容易对齐。

In [ ]:
import torch

x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

row_sum_without_keepdim = x.sum(dim=1)
row_sum_with_keepdim = x.sum(dim=1, keepdim=True)

print("x.shape =", x.shape)
print("x.sum(dim=1).shape =", row_sum_without_keepdim.shape)
print("x.sum(dim=1, keepdim=True).shape =", row_sum_with_keepdim.shape)
print("x.sum(dim=1) =", row_sum_without_keepdim)
print("x.sum(dim=1, keepdim=True) =\n", row_sum_with_keepdim)

### 12.3 `max` 的返回值：最大值和索引

`x.max()` 不指定 `dim` 时，只返回整个 Tensor 中最大的那个数。

`x.max(dim=...)` 指定维度时，会返回两个东西：

| 返回值 | 含义 |
|---|---|
| `values` | 最大值是多少 |
| `indices` | 最大值在该维度上的位置索引 |

这在分类任务里特别常见。模型通常会输出每个类别的分数，我们用 `max(dim=1)` 找到每个样本分数最高的类别。

In [ ]:
import torch

scores = torch.tensor([
    [0.1, 2.5, 0.3],
    [1.2, 0.4, 3.1]
])

print("scores =\n", scores)
print("scores.shape =", scores.shape)
print("整个 Tensor 最大值 scores.max() =", scores.max())

values, indices = scores.max(dim=1)
print("每一行的最大值 values =", values)
print("每一行最大值的位置 indices =", indices)
print("预测类别就是 indices =", indices)

### 12.4 向量点积 dot product 怎么算

向量点积是线性代数里的老朋友：两个同长度向量，对应位置相乘，再把结果加起来。

假设：

$$a = [1, 2, 3]$$

$$b = [4, 5, 6]$$

点积就是：

$$a \cdot b = 1 \times 4 + 2 \times 5 + 3 \times 6 = 32$$

PyTorch 里有三种常见写法：

| 写法 | 说明 | 适合什么时候用 |
|---|---|---|
| `torch.dot(a, b)` | 专门算一维向量点积 | 两个都是 1D Tensor 时最清楚 |
| `(a * b).sum()` | 先逐元素乘法，再求和 | 想看清点积本质时最好理解 |
| `a @ b` | 对一维向量时也是点积 | 和矩阵乘法写法统一 |

注意：`torch.dot` 只适合一维向量。如果是二维矩阵，通常用 `@` 或 `torch.matmul`。

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

dot_1 = torch.dot(a, b)
dot_2 = (a * b).sum()
dot_3 = a @ b

print("torch.dot(a, b):", dot_1)
print("(a * b).sum():", dot_2)
print("a @ b:", dot_3)

print("逐元素乘法 a * b:", a * b)
print("点积结果是一个标量，shape =", dot_1.shape)

### 12.5 点积、逐元素乘法、矩阵乘法别混

| 运算 | 写法 | 输入例子 | 输出 |
|---|---|---|---|
| 逐元素乘法 | `a * b` | `[3]` 和 `[3]` | `[3]` |
| 向量点积 | `torch.dot(a, b)` 或 `a @ b` | `[3]` 和 `[3]` | 标量 `[]` |
| 矩阵乘向量 | `A @ b` | `[2, 3]` 和 `[3]` | `[2]` |
| 矩阵乘矩阵 | `A @ B` | `[2, 3]` 和 `[3, 4]` | `[2, 4]` |

线性层 `nn.Linear` 本质上就和矩阵乘法有关：输入特征和权重做乘法，再加偏置。后面讲 `nn.Module` 时会继续展开。

In [ ]:
A = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])
b = torch.tensor([10.0, 20.0, 30.0])

print("A.shape:", A.shape)
print("b.shape:", b.shape)
print("A @ b:", A @ b)
print("(A @ b).shape:", (A @ b).shape)